# Module 3.1: Homelab Switch Topology, Static IP Subnet Allocation & Tailscale Mesh Unification

## Homelab Expansion (Chapter 2) - Step 3: Multi-Node Networking & Switch Infrastructure

## 🎯 Learning Goals

By completing this laboratory module, you will be able to:

1. Understand why a dedicated **8-Port Gigabit (or 2.5GbE) Switch** is essential to make a multi-node homelab function as a single high-speed unit with room for future expansion.
2. Design a contiguous static IP allocation scheme (`10.0.0.200`–`10.0.0.210`) for current and future homelab nodes.
3. Configure a DHCP MAC address reservation (`10.0.0.201`) for `homelab-gpu` and verify full 1 Gbps wire-speed throughput (`~940 Mbps`) between `homelab-pi` and `homelab-gpu` using `iperf3`.
4. Provision `homelab-gpu` onto your Tailscale WireGuard mesh (applying our Chapter 1 Avahi `deny-interfaces=tailscale0` lesson upfront!).
5. Configure direct node-to-node SSH trust and unified workstation `~/.ssh/config` aliases.

## 3.1.1 Conceptual Background: Engineering a Unified Multi-Node Homelab Network

When your homelab grows from a single Raspberry Pi into a multi-node cluster (`homelab-pi` + `homelab-gpu` + future NAS/mini-PC nodes), plugging devices into random Wi-Fi bands or daisy-chaining consumer router ports creates latency jitter and port exhaustion.

### 1. Why Add a Dedicated Homelab Ethernet Switch?
In Step 4, every photo and video you upload to `homelab-pi` will be sent over the local network to `homelab-gpu` (port `3003`) for NVIDIA CUDA CLIP embedding and facial recognition.
- **Wire-Speed Layer-2 Switching (ASIC Backplane):** By connecting both `homelab-pi` and `homelab-gpu` to a dedicated **8-Port Gigabit Switch** (such as the **TP-Link TL-SG108E** Smart Managed Switch or **Netgear GS308**), all traffic between your Pi and your Zephyrus laptop flows directly across the switch's dedicated hardware ASIC backplane at **940+ Mbps with `<0.3ms` latency**.
- **Zero Router Bottleneck:** Node-to-node traffic never touches your home router's CPU or congests your household Wi-Fi!
- **Single-Uplink Modularity & Future Expansion:** Only **one** Cat 6 cable runs from your homelab switch to your ISP/home router. With an 8-port switch:
  - **Port 1:** Uplink to Home Router
  - **Port 2:** Node 1 — `homelab-pi` (`10.0.0.200`)
  - **Port 3:** Node 2 — `homelab-gpu` (`10.0.0.201`)
  - **Ports 4–8 (Future Expansion):** Ready for a dedicated multi-bay NAS (`10.0.0.202`), a Proxmox virtualization node (`10.0.0.203`), a Raspberry Pi 5 / PoE Zigbee coordinator (`10.0.0.204`), or a laptop diagnostic port.

```mermaid
flowchart TB
    Router["Home Router / Gateway<br>10.0.0.1 (DHCP Server)"] ===|"Port 1 (Single Uplink)"| Switch["8-Port Gigabit Managed Switch<br>(TP-Link TL-SG108E / Netgear GS308)<br>16 Gbps Non-Blocking Backplane"]
    
    Switch ===|"Port 2 (Cat 6)"| Pi["Node 1: homelab-pi<br>LAN: 10.0.0.200<br>Role: NAS + Immich Core + Pi-hole DNS"]
    Switch ===|"Port 3 (Cat 6)"| GPU["Node 2: homelab-gpu (ROG Zephyrus)<br>LAN: 10.0.0.201<br>Role: NVIDIA CUDA ML + Transcoding"]
    Switch -.-|"Ports 4-8 (Reserved)"| Future["Future Nodes (.202 - .206)<br>Proxmox / Backup NAS / Home Assistant"]
```

### 2. Networking Shopping List for Multi-Node Expansion

| Component | Recommended Model | Why It Makes Sense |
| :--- | :--- | :--- |
| **8-Port Gigabit Switch** | **TP-Link TL-SG108E** *(Easy Smart Managed)* or **TL-SG108** *(Unmanaged)* | Fanless steel chassis, IEEE 802.3x flow control, lifetime warranty, and (on the `TL-SG108E`) VLAN and port-mirroring support when you later segment IoT devices. |
| **USB 3.0 to Gigabit / 2.5GbE Adapter** *(if needed)* | **UGREEN / Anker USB 3.0 to RJ45 (`Realtek RTL8153`/`RTL8156`)** | Required only if your 2019 ROG Zephyrus is a `GA502`/`GU502` derivative without a built-in Ethernet port (`GX502` includes a built-in Gigabit RJ45 jack). Plugs into a USB 3.2 Gen 2 port for native Linux kernel plug-and-play. |
| **Color-Coded Cat 6 Patch Cables** | **3-Pack 1ft–3ft Snagless Cat 6 Patch Cables** | Keeps your homelab shelf tidy and makes tracing node connections effortless. |

## 3.1.2 Step-by-Step Laboratory Guide

### Task 3.1.1: Wire the Gigabit Switch & Reserve `10.0.0.201` on Your Router
1. Connect **Port 1** of your new Gigabit switch to a LAN port on your home router.
2. Connect **Port 2** to `homelab-pi` and **Port 3** to `homelab-gpu` using Cat 6 patch cables.
3. On `homelab-gpu`, inspect your active wired Ethernet interface name and MAC (`link/ether`) address:
   ```bash
   ip -br link
   ```
4. Open your home router's administration portal (just as you did in Chapter 1, Module 2.1 for `homelab-pi` at `10.0.0.200`), navigate to **DHCP Static Lease / Address Reservation**, and bind `homelab-gpu`'s Ethernet MAC address to **`10.0.0.201`** (or the next sequential IP in your subnet after `homelab-pi`).
5. Renew the DHCP lease on `homelab-gpu` and verify it holds `10.0.0.201`:
   ```bash
   sudo netplan apply
   ip -4 -br addr show
   ```

### Task 3.1.2: Benchmark Node-to-Node Gigabit Switch Throughput (`iperf3`)
Let's prove that your new switch passes traffic directly between `homelab-pi` and `homelab-gpu` at full Gigabit wire speed (`~940 Mbps`):

1. **On Node 1 (`homelab-pi`):** Install `iperf3` and start it in server listener mode:
   ```bash
   sudo apt update && sudo apt install -y iperf3
   iperf3 -s -1
   ```
2. **On Node 2 (`homelab-gpu`):** Install `iperf3` and run a 5-second bandwidth test against `homelab-pi` (`10.0.0.200`):
   ```bash
   sudo apt install -y iperf3
   iperf3 -c 10.0.0.200 -t 5
   ```
   You should see **`~935 - 945 Mbits/sec`** transfer speeds and sub-millisecond ping latency (`ping -c 4 10.0.0.200`).

### 🖼️ Task 3.1.2 Verification: Node-to-Node Gigabit `iperf3` Benchmark

**Verification Objective:** Capture a screenshot showing the `iperf3 -c 10.0.0.200 -t 5` result on `homelab-gpu` achieving ~940 Mbps across the Gigabit switch.

---

> ### 📷 User Verification Screenshot Submission Box
> *Attach your milestone verification screenshot below (drag & drop an image or use a Markdown image link):*
> 
> ```markdown
> <!-- Insert your screenshot here: -->
> 
> ```

### Task 3.1.3: Join `homelab-gpu` to Tailscale & Apply the Chapter 1 Avahi Fix
Register `homelab-gpu` on your Tailscale WireGuard mesh so you can manage both nodes remotely, and immediately apply our Chapter 1 lesson to prevent `avahi-daemon` from renaming the host on `tailscale0`:

```bash
# 1. Install Tailscale via official script on homelab-gpu
curl -fsSL https://tailscale.com/install.sh | sh

# 2. Authenticate and join your Tailnet
sudo tailscale up

# 3. Apply Chapter 1 Learning: Prevent Avahi mDNS hostname collisions on tailscale0
if [ -f /etc/avahi/avahi-daemon.conf ]; then
    sudo sed -i 's/#deny-interfaces=eth1/deny-interfaces=tailscale0/' /etc/avahi/avahi-daemon.conf
    sudo systemctl restart avahi-daemon
fi

# 4. Verify both homelab-pi and homelab-gpu appear in your Tailnet
tailscale status
```

### Task 3.1.4: Unified Workstation `~/.ssh/config` & Direct Node-to-Node SSH
Update `~/.ssh/config` on your workstation so you can jump into either node seamlessly using its static IP:

```text
Host homelab-pi
    HostName 10.0.0.200
    User pi
    IdentityFile ~/.ssh/id_ed25519_homelab
    IdentitiesOnly yes
    AddKeysToAgent yes

Host homelab-gpu
    HostName 10.0.0.201
    User pi
    IdentityFile ~/.ssh/id_ed25519_homelab
    IdentitiesOnly yes
    AddKeysToAgent yes
```
Verify from your workstation that both `ssh homelab-pi` and `ssh homelab-gpu` connect instantaneously without a password prompt!

### 🖼️ Task 3.1.4 Verification: Unified Tailscale Mesh & Dual-Node SSH Status

**Verification Objective:** Capture a screenshot showing `tailscale status` with both `homelab-pi` and `homelab-gpu` online, along with successful `ssh homelab-gpu` connectivity.

---

> ### 📷 User Verification Screenshot Submission Box
> *Attach your milestone verification screenshot below (drag & drop an image or use a Markdown image link):*
> 
> ```markdown
> <!-- Insert your screenshot here: -->
> 
> ```

## 🏁 Module Checkpoint & Reflection

Before proceeding to the next guide, confirm the following checklist:

- [ ] Both `homelab-pi` (`10.0.0.200`) and `homelab-gpu` (`10.0.0.201`) wired to the Gigabit switch with static DHCP leases.
- [ ] Full Gigabit throughput (`~940 Mbps`) verified between nodes via `iperf3`.
- [ ] `homelab-gpu` joined to Tailscale and `tailscale0` excluded from `avahi-daemon`.
- [ ] Workstation `~/.ssh/config` updated with both `homelab-pi` and `homelab-gpu` host definitions.
- [ ] Verification screenshots captured and attached above.

Proceed to [Module 4.1: NVIDIA Drivers, Container Toolkit & Docker Runtime](../04_gpu_compute_and_workload_migration/04_1_nvidia_drivers_and_container_toolkit.ipynb).